# 04. Делегирование: агент как команда

## Что агент пока не умеет

После `03` у нас есть OpenClaw-подобный контур, но один агент начинает перегружаться контекстом и ролями.

> Subagent — это не ещё одна модель ради красивой схемы. Это способ отделить контекст, специализацию и ответственность.


![Subagents delegation](../visuals/openclaw_path/09_subagents_delegation.svg)


## Роли в этом demo

```text
repo-researcher
Цель: найти факты и релевантные участки кода
Ответственность: read/search и ссылки на файлы
```

```text
analysis-reviewer
Цель: проверить архитектурный отчёт
Ответственность: фактическая точность, пропущенные компоненты, риски
```

В этом demo роли отличаются prompt и отдельным контекстом. В production им можно назначать разные tool sets и permissions.


In [ ]:
from pathlib import Path
import sys

for candidate in (Path.cwd(), Path.cwd() / 'workshop_notebooks' / 'openclaw_path'):
    if (candidate / 'workshop_utils.py').exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

from workshop_utils import REPO_ROOT, print_stage_context, register_graphs, write_text

print_stage_context()


In [ ]:
ENTRYPOINT = "\nfrom __future__ import annotations\n\nimport os\nfrom pathlib import Path\nfrom deepagents import create_deep_agent\nfrom dotenv import load_dotenv\nfrom connectors.jenkins import JENKINS_TOOLS\nfrom connectors.vk import VK_TOOLS\n\nload_dotenv()\nDEFAULT_MODEL = \"openrouter:tencent/hy3:free\"\n\ndef _workspace_root() -> Path:\n    return Path(os.getenv(\"OPENCLAW_WORKSPACE\", \".\")).expanduser().resolve()\n\n\ndef _backend(*, require_shell: bool = False):\n    root = _workspace_root()\n    shell_enabled = os.getenv(\"OPENCLAW_ENABLE_LOCAL_SHELL\") == \"1\"\n    if require_shell and not shell_enabled:\n        raise RuntimeError(\"OPENCLAW_ENABLE_LOCAL_SHELL=1 is required for this stage\")\n    if shell_enabled:\n        from deepagents.backends import LocalShellBackend\n\n        return LocalShellBackend(\n            root_dir=root,\n            virtual_mode=True,\n            inherit_env=False,\n            timeout=120,\n            max_output_bytes=80_000,\n        )\n\n    from deepagents.backends import FilesystemBackend\n\n    return FilesystemBackend(root_dir=root, virtual_mode=True)\n\nTOOLS = [*JENKINS_TOOLS, *VK_TOOLS]\nSUBAGENTS = [\n    {\n        \"name\": \"repo-researcher\",\n        \"description\": \"Trace repository flows and find factual code locations for integration analysis.\",\n        \"system_prompt\": \"You find facts in the repository. Read/search files, cite exact paths, and avoid edits or unsupported claims.\",\n    },\n    {\n        \"name\": \"analysis-reviewer\",\n        \"description\": \"Check factual accuracy, missing components, contradictions, and operational risks in analysis reports.\",\n        \"system_prompt\": \"You verify architecture analysis. Check cited paths, missing components, contradictions, and operational risks. Do not rewrite the whole solution.\",\n    },\n]\nBASE_PROMPT = \"\"\"\\\nYou are OpenClaw at stage 04. Respond in the user's language; default to Russian.\nFor repository integration analysis, delegate to repo-researcher first, then ask analysis-reviewer to check factual accuracy before finalizing.\nIn this demo, subagent roles are separated by prompt and context; production systems can assign separate tool sets and permissions.\n\"\"\"\nagent = create_deep_agent(\n    model=os.getenv(\"OPENCLAW_MODEL\", DEFAULT_MODEL),\n    tools=TOOLS,\n    system_prompt=BASE_PROMPT,\n    subagents=SUBAGENTS,\n    backend=_backend(),\n)\n"
print(write_text("agents/openclaw_04_subagents.py", ENTRYPOINT).relative_to(REPO_ROOT))
print(register_graphs({
    "openclaw_04": "./agents/openclaw_04_subagents.py:agent",
}).relative_to(REPO_ROOT))


## Проверка в LangGraph Studio

### Запрос

```text
Поручи repo-researcher проследить два потока: 1) Jenkins trigger от tool call до HTTP request; 2) VK message от Long Poll до LangGraph и обратно. Для каждого укажи конкретные файлы и точки отказа. Затем попроси analysis-reviewer проверить фактическую точность отчёта.
```

### Ожидаемое поведение

1. Основной агент делегирует `repo-researcher`.
2. Researcher читает `connectors/jenkins.py`, `connectors/vk.py`, `scripts/vk_langgraph_bridge.py`.
3. `analysis-reviewer` проверяет факты и пропуски.
4. Финальный ответ содержит файлы и точки отказа.

### Что изменилось относительно предыдущего этапа

Появилась иерархическая специализация: контекст исследования отделён от финального ответа.

### Текущее ограничение

Делегирование помогает анализировать и проверять, но пока система не замыкает полный цикл изменения проекта.


#TODO: полностью описать добавление субагентов и их полезность